# Quantum Execution Results

This notebook is the experiment orchestrator for the quantum part of the thesis. It does not reimplement mathematical models: MILP, QUBO, annealing, PCE, and IBM adapters are imported from `src/`.

The workflow has two families:

1. **Seren-derived hourly instances**, evaluated across the three representative energy days with MILP, D-Wave simulated annealing, PB-PCE statevector, and an opt-in IBM QPU validation.
2. **Synthetic complex instances**, evaluated with MILP, D-Wave simulated annealing, and PB-PCE statevector only when the full QUBO fits the configured statevector budget.

Every decoded quantum schedule is independently checked for exact-one assignment and GPU capacity. Actual physical energy cost is recalculated from the schedule, renewable availability, and grid prices; raw QUBO energy is never used as the final business metric.

## 1. Shared imports

This section imports reusable functions. The notebook coordinates experiments and presentation only. The key separation is deliberate: the full MILP uses physical renewable/grid dispatch, whereas the reduced QUBO uses an instance-specific effective price and is then re-evaluated under the physical model.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from time import perf_counter

# Support launching Jupyter from either the repository root or notebooks/.
REPO_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / 'src').is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src.config import ModelConfig
from src.data.energy_prices import load_marginalpdbc
from src.data.gpu_unit_adapter import adapt_gpu_unit
from src.data.scenarios import ENERGY_SCENARIOS
from src.data.solar_profiles import build_daily_hourly_solar_profile
from src.milp.runner import run_milp
from src.quantum.annealing import run_simulated_annealing
from src.quantum.benchmarking import (
    add_effective_price_for_qubo,
    evaluate_physical_schedule,
    make_effective_price_qubo_inputs,
)
from src.quantum.pce import (
    ANSATZ_BUILDERS,
    FixedAlphaConfig,
    ProgressiveBinarizationConfig,
    build_statevector_correlation_evaluator,
    evaluate_candidate,
    minimum_pce_qubits,
    pauli_labels_for_qubo,
    run_fixed_alpha,
    run_fixed_alpha_grid,
    run_progressive_binarization,
)
from src.quantum.qubo_builder import QuboPenaltyWeights, build_scheduling_qubo, summarize_qubo_graph

PCE_SELECTION_POLICY = 'best_schedule_feasible_trace_else_final_point'


## 2. Experiment configuration

This section defines a controlled comparison between the full MILP, simulated quantum annealing, and PB-PCE. For each selected workload instance and energy-day profile, all methods are evaluated against the same decoded physical schedule: the comparison therefore distinguishes optimisation performance from the QUBO effective-price approximation.

The configuration fixes the instance scope, energy assumptions, solver stopping rules, QUBO penalty weights, and PCE training procedure. Fixed-alpha PCE runs first identify whether a useful binarization regime exists; PB-PCE then applies progressive binarization using the configured initial alpha. Real-QPU execution is a validation of the selected, statevector-trained circuit and is never used to tune alpha in this notebook.

### Configuration rationale

This cell separates **experimental controls** from **computational limits**. It does not modify the scheduling formulation.

- `SEREN_INSTANCE_NAMES` and `COMPLEX_INSTANCE_NAMES` select the workload families. `RUN_COMPLEX` enables the synthetic stress cases.
- `GPU_UNIT='node'` expresses each demand and capacity in eight-GPU nodes. `PUE`, the renewable price factor, and the energy-day profiles define the physical energy inputs used by the MILP and by decoded-schedule evaluation.
- `MILP_TIME_LIMIT_S` and `MILP_MIP_GAP` define Gurobi's stopping rule. `ANNEALING_READS` is the number of independent samples drawn by the D-Wave simulated annealer.
- `QUBO_WEIGHTS` are the Lagrange multipliers. Assignment has a high weight to enforce one execution per job; GPU capacity is enforced through its squared slack penalty; peak smoothing is disabled for this baseline comparison.
- `PCE_ANSATZ_ID`, `PCE_LAYERS`, optimizer budget (`maxiter`), alpha grid, and PB-PCE configuration define the gate-model experiment.

#### Why statevector simulation is capped at 16 PCE qubits

`MAX_STATEVECTOR_PCE_QUBITS` applies **only** to the local classical statevector simulator. A circuit with $n$ qubits has one complex amplitude per computational-basis state:

$$
N_{amplitudes} = 2^n
$$

A `complex128` amplitude requires 16 bytes, hence the raw statevector storage is:

$$
M_{raw} = 16 \cdot 2^n \text{ bytes}
$$

| PCE qubits $n$ | Complex amplitudes $2^n$ | Raw `complex128` state memory |
|---:|---:|---:|
| 16 | 65,536 | about 1 MiB |
| 24 | 16,777,216 | about 256 MiB |
| 32 | 4,294,967,296 | about 64 GiB |
| 135 | about $4.4 \times 10^{40}$ | about $7.0 \times 10^{41}$ bytes |

PCE optimisation evaluates the circuit repeatedly across parameter values and measurement groups. Sixteen qubits is therefore a conservative local-development budget, not a claim about PCE capability. With an order-two encoding, $n$ PCE qubits represent at most:

$$
M_{QUBO,max} = 3 \cdot \frac{n(n-1)}{2}
$$

Thus, 16 PCE qubits can represent 360 QUBO variables.

For IBM execution, this statevector cap is not used. The notebook derives the PCE width required by the QUBO and selects a backend with at least that many physical qubits. A 135-qubit device has a theoretical order-two representation capacity of:

$$
3 \cdot \frac{135 \cdot 134}{2} = 27{,}135
$$

QUBO variables. In practice, usable problem size is governed by transpilation, hardware connectivity, noise, circuit depth, shot budget, and optimizer evaluations rather than statevector memory.

In [16]:
ROOT = REPO_ROOT
SEREN_ROOT = ROOT / 'data' / 'instances' / 'seren_pretrain'
SOLAR_PATH = ROOT / 'data' / 'energy' / 'solar' / 'Timeseries_40.409_-3.698_SA3_1000kWp_crystSi_14_36deg_-5deg_2020_2023.csv'
RESULTS_DIR = ROOT / 'data' / 'results' / 'quantum_execution'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEREN_INSTANCE_NAMES = ('tiny_6_jobs','small_12_jobs','medium_24_jobs','avg_active_day_jobs','max_day_jobs')
COMPLEX_INSTANCE_NAMES = ('synthetic_duration_dense_2_to_4h_seed_7101',)
RUN_COMPLEX = True
RUN_PCE_QPU = False

GPU_UNIT = 'node'
PUE = 1.25
RENEWABLE_PRICE_FACTOR = 0.80
MILP_TIME_LIMIT_S = 60
MILP_MIP_GAP = 0.001
ANNEALING_READS = 1_000
ANNEALING_SEED = 123

QUBO_WEIGHTS = QuboPenaltyWeights(
    assignment=1_000.0,
    gpu_capacity=1.0,
    peak_smoothing=0.0,
)
PCE_ANSATZ_ID = 'legacy_chain_ry_rz_cx_v1'
PCE_LAYERS = 1
PCE_ALPHA_GRID = (0.5, 1.0, 2.0, 4.0, 8.0)
PCE_SEED = 2026
PCE_FIXED_CONFIG = FixedAlphaConfig(maxiter=120)
PB_CONFIG = ProgressiveBinarizationConfig(
    initial_alpha=3.0,
    binarization_target=0.90,
    max_stages=12,
    fixed_alpha=PCE_FIXED_CONFIG,
)
# A statevector stores 2**n amplitudes, so this limit is PCE circuit width rather
# than raw QUBO size. At order two, 16 PCE qubits encode 3 * C(16, 2) = 360 variables.
MAX_STATEVECTOR_PCE_QUBITS = 16

assert PCE_ANSATZ_ID in ANSATZ_BUILDERS


## 3. Shared data preparation

This section creates the common workload and energy inputs used in every solver comparison. The aim is to hold the job set, cluster capacity, PUE, grid-price day, and solar-generation profile constant, while making explicit the unavoidable modelling difference between the full MILP and the reduced QUBO.

### Workload and resource units

Every instance is converted to `GPU_UNIT='node'`: one resource unit is one eight-GPU A100 node. Job GPU demand and cluster GPU capacity are converted with the same rule, so the MILP and QUBO solve the same discrete node-allocation problem. CPU and memory columns are retained for the MILP data model, although the reduced QUBO only enforces assignment and GPU capacity.

### Physical energy profile

For each energy-day scenario, `build_energy_day` joins:

- hourly day-ahead grid prices;
- an hourly PVGIS solar-generation profile for the selected date;
- a renewable price set to `RENEWABLE_PRICE_FACTOR` times that day's minimum grid price;
- the fixed PUE and zero baseline facility load used in this experiment.

This produces the physical profile used by the MILP and by the final evaluation of every decoded schedule. The physical evaluator dispatches renewable supply first whenever it is no more expensive than grid electricity; remaining facility demand is grid import. Therefore, reported physical cost, renewable energy, grid energy, and peak grid import are comparable across all methods.

### Why the QUBO uses an effective price

The full MILP has explicit renewable and grid variables in every hour. It can decide how much facility load is supplied by each source, subject to renewable availability. The reduced QUBO has only binary job-start variables and GPU-capacity slack bits; it has no variables for renewable dispatch or grid import.

Before QUBO construction, the notebook derives an instance-specific hourly `effective_price`. It estimates a representative facility-load scale from the instance's total job energy divided by the horizon, with a floor equal to 10% of total facility capacity. Solar availability divided by that scale gives a renewable share between zero and one. The QUBO price is then the weighted average of renewable and grid price using that share. High solar availability therefore creates a lower coefficient for scheduling load in that hour, without adding dispatch variables to the QUBO.

The QUBO builder receives this effective price as its hourly energy coefficient and deliberately receives zero renewable availability. This prevents its simpler binary renewable-availability rule from changing the continuous effective-price signal. This is an optimisation approximation only: after a QUBO schedule is decoded, its real energy cost is recalculated using the physical renewable-first dispatch described above.

### Peak treatment

The full MILP represents the physical grid-import peak. It defines `P_peak` as the maximum hourly grid import, then charges a penalty on `max(P_peak - contracted_power, 0)`. In this notebook, `peak_price=1000`, but `contracted_power` is set to aggregate facility capacity and baseline load is zero. Because resource capacity already bounds scheduled load, the excess-peak term will normally be inactive. It remains in the MILP configuration so that a future experiment can activate it simply by selecting a lower contracted-power limit.

The QUBO cannot encode the MILP max-grid-import term directly without additional binary variables and higher complexity. Its available approximation is a load-smoothing term proportional to the sum of squared hourly facility loads. This discourages concentrated demand but is not equivalent to contracted-power billing and does not distinguish renewable from grid supply. Here `peak_smoothing=0`, so the QUBO also has no active peak objective. Consequently, the current results isolate energy scheduling and constraint feasibility; they must not be interpreted as a peak-optimisation comparison.


In [17]:
def load_instance(instance_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load a Seren or synthetic instance and select the common GPU unit."""

    instance_dir = SEREN_ROOT / instance_name
    jobs_raw = pd.read_csv(instance_dir / 'jobs.csv')
    clusters_raw = pd.read_csv(instance_dir / 'clusters.csv')
    jobs, clusters = adapt_gpu_unit(jobs_raw, clusters_raw, GPU_UNIT)
    jobs['cpu_required'] = jobs.get('cpu_required', pd.Series(0.0, index=jobs.index)).fillna(0.0)
    jobs['memory_required_gb'] = jobs.get('memory_required_gb', pd.Series(0.0, index=jobs.index)).fillna(0.0)
    return jobs, clusters


def build_energy_day(name: str, scenario: dict) -> pd.DataFrame:
    """Build one physical hourly energy profile from price and solar inputs."""

    prices = load_marginalpdbc(ROOT / scenario['price_file'], price_column='last')
    solar = build_daily_hourly_solar_profile(SOLAR_PATH, scenario['date'])
    energy = prices[['hour', 'grid_price']].merge(
        solar[['hour', 'renewable_available']], on='hour', how='inner', validate='one_to_one'
    )
    energy['renewable_price'] = RENEWABLE_PRICE_FACTOR * float(energy['grid_price'].min())
    energy['pue'] = PUE
    energy['baseline_load'] = 0.0
    energy['scenario'] = name
    return energy


ENERGY_DAYS = {name: build_energy_day(name, scenario) for name, scenario in ENERGY_SCENARIOS.items()}
energy_day_summary = pd.DataFrame([
    {
        'energy_day': name,
        'min_grid_price': day.grid_price.min(),
        'mean_grid_price': day.grid_price.mean(),
        'max_grid_price': day.grid_price.max(),
        'renewable_energy_mwh': day.renewable_available.sum(),
    }
    for name, day in ENERGY_DAYS.items()
])
display(energy_day_summary.round(4))


,energy_day,min_grid_price,mean_grid_price,max_grid_price,renewable_energy_mwh
0,clear_sky,8.00,56.2054,106.14,6.5620
1,overcast,65.70,87.1600,107.36,1.0555
2,base,77.14,114.8346,170.00,5.0676


## 4. Shared execution helpers

This section defines the reusable functions that run one instance/day pair and convert each solver output into a comparable result row. `run_case` is used later by both the Seren and synthetic-complex experiments.

### Configuration helpers

- `milp_config(clusters, energy)` builds the `ModelConfig` for the full formulation. It passes the physical renewable availability, grid prices, renewable price, PUE, and contracted-power limit to Gurobi. The contracted limit is currently aggregate facility capacity.
- `qubo_config(clusters, energy)` builds the reduced-model configuration. The QUBO receives precomputed effective hourly prices rather than physical renewable dispatch inputs. Its peak-billing coefficient is zero because the QUBO peak proxy is controlled separately through `QUBO_WEIGHTS.peak_smoothing`.

### Evaluation and reporting helpers

- `physical_metrics(schedule, energy)` re-evaluates a decoded schedule using physical renewable-first dispatch. It returns total energy cost, renewable and grid energy, facility energy, and maximum grid import. This creates a common basis for MILP, annealing, and PCE results.
- `result_row(...)` standardises solver outputs into one record. Besides feasibility and runtime, it stores QUBO size, quadratic-coupler count, PCE circuit width, selected alpha, constraint violations, and physical energy metrics. A QUBO Hamiltonian energy is retained for diagnostics but is not used as the cross-method business metric.

### PCE and end-to-end execution helpers

- `run_pce_statevector(qubo, jobs, clusters)` maps a QUBO to order-two Pauli Correlation Encoding, checks that its **PCE qubit width** fits the local statevector budget, constructs the selected ansatz, scans fixed alpha values, and then runs PB-PCE. It returns the full calibration and PB-PCE artefacts rather than only a schedule.
- `run_case(instance_name, energy_day_name, jobs, clusters, energy)` is the orchestrator for one comparable experiment. It runs Gurobi on physical inputs, builds the effective-price QUBO, samples that QUBO with the D-Wave simulated annealer, runs statevector PB-PCE when allowed, decodes each result, validates job assignment and GPU capacity, and produces comparison rows.

The QPU path is intentionally separate from `run_case`: it is optional, consumes external allocation, and validates a selected statevector-trained PCE circuit rather than tuning its parameters on hardware.

PCE is not run merely because its Pauli compression fits. Statevector simulation is capped by the required PCE qubits because its memory grows exponentially with circuit width. IBM execution does not inherit that cap: it derives the required PCE width from the QUBO and validates it against the selected backend. A skipped statevector result is reported as a scaling limit, not treated as failure.

In [18]:
def milp_config(clusters: pd.DataFrame, energy: pd.DataFrame) -> ModelConfig:
    """Return full-MILP configuration for one physical energy day."""

    facility_capacity = float(clusters['capacity'].sum()) * float(energy['pue'].iloc[0])
    return ModelConfig(
        contracted_power=facility_capacity,
        renewable_price=float(energy['renewable_price'].iloc[0]),
        peak_price=1_000.0,
        delta_t=1.0,
        pue=float(energy['pue'].iloc[0]),
    )


def qubo_config(clusters: pd.DataFrame, energy: pd.DataFrame) -> ModelConfig:
    """Return reduced-QUBO configuration using precomputed effective prices."""

    return ModelConfig(
        contracted_power=float(clusters['capacity'].sum()) * float(energy['pue'].iloc[0]),
        renewable_price=1_000_000.0,
        peak_price=0.0,
        delta_t=1.0,
        pue=float(energy['pue'].iloc[0]),
    )


def physical_metrics(schedule: pd.DataFrame, energy: pd.DataFrame) -> dict[str, float | None]:
    """Return comparable physical energy metrics only for a decoded schedule."""

    if schedule.empty:
        return {
            'physical_total_cost': None,
            'grid_energy_mwh': None,
            'renewable_energy_mwh': None,
            'facility_energy_mwh': None,
            'peak_grid_import_mw': None,
        }
    _, metrics = evaluate_physical_schedule(schedule, energy, delta_t=1.0)
    return {
        'physical_total_cost': metrics['physical_total_cost'],
        'grid_energy_mwh': metrics['grid_energy_mwh'],
        'renewable_energy_mwh': metrics['renewable_energy_mwh'],
        'facility_energy_mwh': metrics['facility_energy_mwh'],
        'peak_grid_import_mw': metrics['physical_peak_grid_import_mw'],
    }


def result_row(
    *, instance: str, energy_day: str, method: str, schedule: pd.DataFrame,
    validation: dict, runtime_s: float | None, qubo=None, qubo_energy: float | None = None,
    pce_qubits: int | None = None, ansatz: str | None = None, alpha: float | None = None,
    pb_stages: int | None = None, status: str = 'completed', extra: dict | None = None,
) -> dict:
    """Normalize solver outputs into a comparison-table row."""

    feasible = bool(validation.get('feasible', False))
    row = {
        'instance': instance,
        'energy_day': energy_day,
        'method': method,
        'status': status,
        'feasible': feasible,
        'runtime_s': runtime_s,
        'qubo_energy': qubo_energy,
        'assignment_variables': (qubo.metadata.get('num_assignment_variables') if qubo else None),
        'slack_variables': (qubo.metadata.get('num_slack_variables') if qubo else None),
        'qubo_variables': (qubo.num_variables if qubo else None),
        'quadratic_couplers': (len(qubo.quadratic) if qubo else None),
        'pce_qubits': pce_qubits,
        'ansatz': ansatz,
        'alpha': alpha,
        'pb_stages': pb_stages,
        'violation_count': len(validation.get('violations', [])),
        'violations': '; '.join(validation.get('violations', [])),
    }
    row.update(physical_metrics(schedule, energy) if feasible else physical_metrics(pd.DataFrame(), energy))
    if extra:
        row.update(extra)
    return row


def run_pce_statevector(qubo, jobs: pd.DataFrame, clusters: pd.DataFrame) -> dict:
    """Calibrate fixed alpha, then execute PB-PCE on one reduced QUBO."""

    required_qubits = minimum_pce_qubits(qubo.num_variables, order=2)
    if required_qubits > MAX_STATEVECTOR_PCE_QUBITS:
        return {
            'status': 'skipped_statevector_qubits',
            'reason': (
                f'QUBO maps to {required_qubits} PCE qubits; statevector budget is '
                f'{MAX_STATEVECTOR_PCE_QUBITS}'
            ),
            'qubits': required_qubits,
        }
    qubits, labels = pauli_labels_for_qubo(qubo, order=2)
    specification = ANSATZ_BUILDERS[PCE_ANSATZ_ID]
    circuit, parameters = specification['builder'](qubits, PCE_LAYERS)
    evaluator = build_statevector_correlation_evaluator(circuit, parameters, labels)
    initial_theta = np.random.default_rng(PCE_SEED).uniform(-np.pi, np.pi, len(parameters))
    started = perf_counter()
    alpha_runs, alpha_selected = run_fixed_alpha_grid(
        PCE_ALPHA_GRID, initial_theta, evaluator, qubo, jobs, clusters,
        qubo.metadata['hours'], PCE_FIXED_CONFIG,
    )
    pb_run = run_progressive_binarization(
        initial_theta, evaluator, qubo, jobs, clusters, qubo.metadata['hours'], PB_CONFIG,
    )
    return {
        'status': 'completed',
        'runtime_s': perf_counter() - started,
        'qubits': qubits,
        'labels': labels,
        'circuit': circuit,
        'parameters': parameters,
        'alpha_runs': alpha_runs,
        'alpha_selected': alpha_selected,
        'pb_run': pb_run,
    }


def run_case(instance_name: str, energy_day_name: str, jobs: pd.DataFrame, clusters: pd.DataFrame, energy: pd.DataFrame) -> tuple[list[dict], dict]:
    """Run MILP, simulated annealing, and PB-PCE statevector for one case."""

    rows: list[dict] = []
    milp = run_milp(
        jobs, energy[['hour', 'renewable_available', 'grid_price', 'pue', 'baseline_load']], clusters,
        milp_config(clusters, energy), time_limit=MILP_TIME_LIMIT_S, mip_gap=MILP_MIP_GAP,
        model_name=f'quantum_execution_{instance_name}_{energy_day_name}',
    )
    milp_validation = {'feasible': True, 'violations': []}
    rows.append(result_row(
        instance=instance_name, energy_day=energy_day_name, method='MILP_Gurobi',
        schedule=milp.schedule, validation=milp_validation, runtime_s=milp.runtime_s,
        extra={'milp_objective': milp.objective, 'mip_gap': milp.mip_gap, 'solver_status': milp.status},
    ))

    energy_with_effective_price = add_effective_price_for_qubo(energy, jobs, clusters)
    qubo_inputs = make_effective_price_qubo_inputs(energy_with_effective_price)
    qubo = build_scheduling_qubo(jobs, qubo_inputs, clusters, qubo_config(clusters, energy), QUBO_WEIGHTS)
    annealing = run_simulated_annealing(
        qubo, jobs, clusters, qubo.metadata['hours'], num_reads=ANNEALING_READS, seed=ANNEALING_SEED,
    )
    rows.append(result_row(
        instance=instance_name, energy_day=energy_day_name, method='DWave_simulated_annealing',
        schedule=annealing.schedule, validation=annealing.validation, runtime_s=annealing.runtime_s,
        qubo=qubo, qubo_energy=annealing.qubo_energy,
        extra={'num_reads': annealing.num_reads, 'feasible_sample_count': annealing.feasible_sample_count},
    ))

    pce = run_pce_statevector(qubo, jobs, clusters)
    if pce['status'] == 'completed':
        candidate = pce['pb_run'].selected_candidate
        rows.append(result_row(
            instance=instance_name, energy_day=energy_day_name, method='PB_PCE_statevector',
            schedule=candidate.schedule, validation=candidate.validation, runtime_s=pce['runtime_s'],
            qubo=qubo, qubo_energy=candidate.qubo_energy, pce_qubits=pce['qubits'],
            ansatz=PCE_ANSATZ_ID, alpha=candidate.alpha,
            pb_stages=len(pce['pb_run'].stages),
            extra={'fixed_alpha_selected': pce['alpha_selected'].alpha},
        ))
    else:
        rows.append(result_row(
            instance=instance_name, energy_day=energy_day_name, method='PB_PCE_statevector',
            schedule=pd.DataFrame(), validation={'feasible': False, 'violations': [pce['reason']]},
            runtime_s=None, qubo=qubo, status=pce['status'], ansatz=PCE_ANSATZ_ID,
        ))
    return rows, {'qubo': qubo, 'pce': pce, 'energy': energy, 'jobs': jobs, 'clusters': clusters}


## 4.1 Fixed-alpha PCE diagnostic

This short check makes the fixed-alpha execution path observable before the full multi-instance experiment. It builds the same node-level QUBO used by the Seren comparison for `tiny_6_jobs` and `clear_sky`, then runs only `alpha=0.5`.

The result shows the QUBO size and whether a feasible schedule appeared during COBYLA's search, rather than only at its final point.

A `True` value for `selected_from_feasible_trace` means the optimizer visited a schedule-feasible point even if its final numerical point was infeasible. Schedule feasibility validates one execution of every job and GPU capacity; it does not validate the auxiliary slack-bit equality.


In [ ]:
DIAGNOSTIC_INSTANCE = 'tiny_6_jobs'
DIAGNOSTIC_ENERGY_DAY = 'clear_sky'
DIAGNOSTIC_ALPHA = 0.5

diagnostic_jobs, diagnostic_clusters = load_instance(DIAGNOSTIC_INSTANCE)
diagnostic_energy = ENERGY_DAYS[DIAGNOSTIC_ENERGY_DAY]
diagnostic_qubo_energy = add_effective_price_for_qubo(
    diagnostic_energy, diagnostic_jobs, diagnostic_clusters
)
diagnostic_qubo = build_scheduling_qubo(
    diagnostic_jobs,
    make_effective_price_qubo_inputs(diagnostic_qubo_energy),
    diagnostic_clusters,
    qubo_config(diagnostic_clusters, diagnostic_energy),
    QUBO_WEIGHTS,
)
diagnostic_qubits, diagnostic_labels = pauli_labels_for_qubo(diagnostic_qubo, order=2)
diagnostic_circuit, diagnostic_parameters = ANSATZ_BUILDERS[PCE_ANSATZ_ID]['builder'](
    diagnostic_qubits, PCE_LAYERS
)
diagnostic_evaluator = build_statevector_correlation_evaluator(
    diagnostic_circuit, diagnostic_parameters, diagnostic_labels
)
diagnostic_theta = np.random.default_rng(PCE_SEED).uniform(
    -np.pi, np.pi, len(diagnostic_parameters)
)
diagnostic_run = run_fixed_alpha(
    DIAGNOSTIC_ALPHA, diagnostic_theta, diagnostic_evaluator, diagnostic_qubo,
    diagnostic_jobs, diagnostic_clusters, diagnostic_qubo.metadata['hours'],
    PCE_FIXED_CONFIG,
)
diagnostic_candidate = diagnostic_run.selected_candidate

diagnostic_summary = pd.DataFrame([{
    'instance': DIAGNOSTIC_INSTANCE,
    'energy_day': DIAGNOSTIC_ENERGY_DAY,
    'alpha': DIAGNOSTIC_ALPHA,
    'selection_policy': PCE_SELECTION_POLICY,
    'qubo_variables': diagnostic_qubo.num_variables,
    'quadratic_couplers': len(diagnostic_qubo.quadratic),
    'pce_qubits': diagnostic_qubits,
    'optimizer_evaluations': len(diagnostic_run.history),
    'selected_from_feasible_trace': diagnostic_run.selected_from_feasible_trace,
    'schedule_feasible': diagnostic_candidate.validation['feasible'],
    'qubo_energy': diagnostic_candidate.qubo_energy,
    'pce_loss': diagnostic_candidate.pce_loss,
    'violations': '; '.join(diagnostic_candidate.validation['violations']),
}])
display(diagnostic_summary)
print(
    'Feasible evaluations in the 120-evaluation trace:',
    int(diagnostic_run.history['physical_feasible'].sum()),
)
display(diagnostic_run.history.head(5))


## 5. Seren-derived instances

For each Seren instance and energy day, this section runs the classical reference, the D-Wave simulated-annealing baseline, and PB-PCE in exact statevector simulation. The result table is the main preliminary comparison: only feasible rows should be compared by physical total cost.

The fixed-alpha calibration table immediately after the main comparison shows why PB-PCE uses its configured initial alpha and whether any fixed-alpha schedule is feasible.

The following QPU subsection is opt-in. It first evaluates the statevector-selected PB-PCE circuit on IBM hardware. This is a hardware-noise validation, not yet hardware-in-the-loop training. Full hardware PB-PCE is kept separate because every optimizer evaluation submits QPU work.

In [ ]:
seren_rows: list[dict] = []
seren_artifacts: dict[tuple[str, str], dict] = {}
for instance_name in SEREN_INSTANCE_NAMES:
    jobs, clusters = load_instance(instance_name)
    for energy_day_name, energy in ENERGY_DAYS.items():
        rows, artifacts = run_case(instance_name, energy_day_name, jobs, clusters, energy)
        seren_rows.extend(rows)
        seren_artifacts[(instance_name, energy_day_name)] = artifacts

seren_results = pd.DataFrame(seren_rows)
seren_results.to_csv(RESULTS_DIR / 'seren_quantum_execution_results.csv', index=False)
display(seren_results.sort_values(['instance', 'energy_day', 'method']))

pce_alpha_rows = []
for (instance_name, energy_day_name), artifacts in seren_artifacts.items():
    pce = artifacts['pce']
    if pce['status'] != 'completed':
        continue
    for alpha_run in pce['alpha_runs']:
        candidate = alpha_run.selected_candidate
        pce_alpha_rows.append({
            'instance': instance_name,
            'energy_day': energy_day_name,
            'alpha': alpha_run.alpha,
            'feasible': candidate.validation['feasible'],
            'qubo_energy': candidate.qubo_energy,
            'pce_loss': candidate.pce_loss,
            'optimizer_evaluations': len(alpha_run.history),
            'violation_count': len(candidate.validation['violations']),
            'selected_from_feasible_trace': alpha_run.selected_from_feasible_trace,
            'selection_policy': PCE_SELECTION_POLICY,
        })
pce_alpha_calibration = pd.DataFrame(pce_alpha_rows)
if not pce_alpha_calibration.empty:
    display(pce_alpha_calibration.sort_values(['instance', 'energy_day', 'alpha']))


### 5.1 IBM QPU validation of PB-PCE

This section is disabled by default. When enabled, it submits the PB-PCE statevector-selected parameters to IBM and decodes the measured correlations with the same alpha. The hardware result therefore answers: *does noise preserve the statevector schedule?*

It does not tune alpha on hardware. Hardware-in-the-loop PB-PCE should be activated only after this comparison is stable, because it requires repeated QPU evaluations during optimization.

In [6]:
qpu_rows: list[dict] = []
if RUN_PCE_QPU:
    from dotenv import load_dotenv
    from src.quantum.pce_ibm import (
        build_ibm_correlation_evaluator,
        create_ibm_service,
        select_qpu_backend,
    )

    load_dotenv(ROOT / '.env', override=False)
    service = create_ibm_service(os.environ['QISKIT_IBM_TOKEN'], os.getenv('QISKIT_IBM_INSTANCE'))
    for (instance_name, energy_day_name), artifacts in seren_artifacts.items():
        pce = artifacts['pce']
        if pce['status'] != 'completed':
            continue
        candidate = pce['pb_run'].selected_candidate
        if not candidate.validation['feasible']:
            print(f'Skipping {instance_name}/{energy_day_name}: PB-PCE statevector result is infeasible.')
            continue
        backend = select_qpu_backend(service, pce['qubits'], os.getenv('IBM_BACKEND_NAME') or None)
        evaluator = build_ibm_correlation_evaluator(backend, pce['circuit'], pce['labels'])
        hardware_candidate = evaluate_candidate(
            candidate.theta, candidate.alpha, evaluator, artifacts['qubo'], artifacts['jobs'],
            artifacts['clusters'], artifacts['qubo'].metadata['hours'],
        )
        qpu_rows.append(result_row(
            instance=instance_name, energy_day=energy_day_name, method='PB_PCE_IBM_QPU',
            schedule=hardware_candidate.schedule, validation=hardware_candidate.validation,
            runtime_s=None, qubo=artifacts['qubo'], qubo_energy=hardware_candidate.qubo_energy,
            pce_qubits=pce['qubits'], ansatz=PCE_ANSATZ_ID, alpha=candidate.alpha,
            pb_stages=len(pce['pb_run'].stages),
            extra={
                'backend': backend.name,
                'transpiled_depth': evaluator.isa_ansatz.depth(),
                'job_id': evaluator.last_job_id,
                'mean_absolute_correlation_error': float(np.mean(np.abs(hardware_candidate.correlations - candidate.correlations))),
            },
        ))

qpu_results = pd.DataFrame(qpu_rows)
if not qpu_results.empty:
    qpu_results.to_csv(RESULTS_DIR / 'seren_ibm_qpu_results.csv', index=False)
    display(qpu_results)
else:
    print('No IBM QPU jobs submitted. Set RUN_PCE_QPU=True only after reviewing the statevector rows.')


No IBM QPU jobs submitted. Set RUN_PCE_QPU=True only after reviewing the statevector rows.


## 6. Synthetic complex instances

Complex instances are designed to increase combinatorial difficulty through multi-node packing, overlapping start windows, and competition for low-cost periods. Gurobi and simulated annealing are always attempted when this section is enabled. PB-PCE is attempted only below the statevector QUBO-variable limit and otherwise produces an explicit `skipped_qubo_size` row.

This separates two claims: a problem can be difficult for classical optimization while still exceeding current gate-based PCE simulation limits.

In [14]:
complex_rows: list[dict] = []
if RUN_COMPLEX:
    for instance_name in COMPLEX_INSTANCE_NAMES:
        jobs, clusters = load_instance(instance_name)
        for energy_day_name, energy in ENERGY_DAYS.items():
            rows, _ = run_case(instance_name, energy_day_name, jobs, clusters, energy)
            complex_rows.extend(rows)

complex_results = pd.DataFrame(complex_rows)
if not complex_results.empty:
    complex_results.to_csv(RESULTS_DIR / 'complex_quantum_execution_results.csv', index=False)
    display(complex_results.sort_values(['instance', 'energy_day', 'method']))
else:
    print('Complex experiments are disabled. Set RUN_COMPLEX=True after confirming the Seren baseline.')


,instance,energy_day,method,status,feasible,runtime_s,qubo_energy,assignment_variables,slack_variables,qubo_variables,...,physical_total_cost,grid_energy_mwh,renewable_energy_mwh,facility_energy_mwh,peak_grid_import_mw,milp_objective,mip_gap,solver_status,num_reads,feasible_sample_count
7,synthetic_duration_dense_2_to_4h_seed_7101,base,DWave_simulated_annealing,completed,False,29.035547,1900.330574,808.0,628.0,1436.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1000.0,0.0
6,synthetic_duration_dense_2_to_4h_seed_7101,base,MILP_Gurobi,completed,True,0.359027,NaN,NaN,NaN,NaN,...,842.612477,5.84837,5.06763,10.916,1.04945,842.612477,0.000900,2.0,NaN,NaN
8,synthetic_duration_dense_2_to_4h_seed_7101,base,PB_PCE_statevector,skipped_statevector_qubits,False,NaN,NaN,808.0,628.0,1436.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,synthetic_duration_dense_2_to_4h_seed_7101,clear_sky,DWave_simulated_annealing,completed,False,28.522738,1334.527340,808.0,628.0,1436.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1000.0,0.0
0,synthetic_duration_dense_2_to_4h_seed_7101,clear_sky,MILP_Gurobi,completed,True,0.636239,NaN,NaN,NaN,NaN,...,102.296260,4.36870,6.54730,10.916,1.11097,102.296260,0.000000,2.0,NaN,NaN
2,synthetic_duration_dense_2_to_4h_seed_7101,clear_sky,PB_PCE_statevector,skipped_statevector_qubits,False,NaN,NaN,808.0,628.0,1436.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,synthetic_duration_dense_2_to_4h_seed_7101,overcast,DWave_simulated_annealing,completed,True,28.754141,963.502913,808.0,628.0,1436.0,...,921.381991,9.86049,1.05551,10.916,0.81612,NaN,NaN,NaN,1000.0,1.0
3,synthetic_duration_dense_2_to_4h_seed_7101,overcast,MILP_Gurobi,completed,True,0.047039,NaN,NaN,NaN,NaN,...,843.835031,9.86049,1.05551,10.916,1.05562,843.835031,0.000923,2.0,NaN,NaN
5,synthetic_duration_dense_2_to_4h_seed_7101,overcast,PB_PCE_statevector,skipped_statevector_qubits,False,NaN,NaN,808.0,628.0,1436.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Interpretation checklist

Use the results tables in this order:

1. Filter to `feasible == True` before comparing physical total cost.
2. Compare Gurobi runtime and MIP gap with annealing/PCE runtime only as an experiment-level indicator; queue time and classical optimizer evaluations are reported separately.
3. Compare QUBO variable/coupler count with PCE qubit count. Qubit compression does not remove the classical optimization or measurement burden.
4. Treat an IBM-QPU row as a hardware-noise experiment, not evidence of quantum advantage.
5. Treat `skipped_qubo_size` as a documented current PCE/statevector boundary rather than solver failure.